# Fine-tuning Notebook

Generated by LLM Architecture Explorer.

**Base model:** `/media/amiyaun/New Volume/chai-projects/llama/models/llama-3.2-1B-Instruct`
**Task:** Causal language modeling (fine-tuning)

This notebook was assembled automatically based on the architecture,
pruning, and PEFT configuration selected in the explorer. All cells are
fully editable — nothing here is locked.


## Resource estimate (from explorer, at generation time)

| Metric | Value |
|---|---|
| Total params | 1,235,814,400 |
| Active params (after pruning) | 1,235,814,400 |
| Trainable params | 1,703,936 |
| Trainable % | 0.138% |
| Estimated VRAM (params + optimizer, excl. activations) | 2.49 GB |

These are estimates computed from the model's declared architecture and your
selected configuration — actual usage will vary with batch size, sequence
length, and dataset.


In [ ]:
# Dependencies
# !pip install transformers datasets peft accelerate bitsandbytes trl


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer


In [ ]:
# Load base model and tokenizer
MODEL_PATH = '/media/amiyaun/New Volume/chai-projects/llama/models/llama-3.2-1B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,   # adjust to your hardware (float16 / float32 as needed)
    device_map="auto",
)


In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05000000074505806,
    target_modules=['k_proj', 'o_proj', 'q_proj', 'v_proj'],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:
# ---------------------------------------------------------------
# Dataset loading — fill this in.
#
# Example:
#   dataset = load_dataset("your_dataset_name", split="train")
#
# The dataset should have a text field the tokenizer can consume, or you
# can define a formatting function and pass it to SFTTrainer below via
# `formatting_func`.
# ---------------------------------------------------------------

dataset = None  # TODO: load your dataset here


In [ ]:
training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,   # adjust to your hardware
    report_to="none",
)


In [ ]:
# ---------------------------------------------------------------
# NOTE: requires `dataset` to be set above (currently a placeholder).
# ---------------------------------------------------------------
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()


In [ ]:
# Save adapter (if PEFT was used) or full model
trainer.save_model("./output/final")
tokenizer.save_pretrained("./output/final")

# If using PEFT and you want a merged, standalone model:
# merged_model = model.merge_and_unload()
# merged_model.save_pretrained("./output/merged")
